# 05 — Analiz, Grafikler ve İstatistik (İP-7, İP-8)

Deney klasörlerindeki özet/soru-bazında CSV'lerden rapor tabloları ve grafikler üretir. Bu notebook GPU gerektirmez.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

REPO_URL = 'https://github.com/EmreK37/RAG-Tabanli-Turkce-Soru-Cevap-Sistemleri.git'
BRANCH = 'RAG-dev'
REPO_DIR = '/content/RAG-proje'
if os.path.exists('/content'):
    if not os.path.exists(REPO_DIR):
        !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}
    !git pull origin {BRANCH}

klasorler = sorted(Path('results').glob('ana-karsilastirma-*'))
son = klasorler[-1]
ozet = pd.read_csv(son / 'ozet.csv')
soru_yolu = son / 'soru-bazinda.csv'
soru = pd.read_csv(soru_yolu) if soru_yolu.exists() else None
print('Deney:', son)
ozet

In [ ]:
# Sistem karşılaştırma grafiği (rapor için PNG kaydeder)
fig, ax = plt.subplots(figsize=(8, 4))
ozet.plot.bar(x='sistem', y=['recall@10', 'ndcg@10', 'mrr@10'], ax=ax)
ax.set_ylabel('Skor')
ax.set_title('Retrieval karşılaştırması')
plt.tight_layout()
plt.savefig(son / 'grafik-retrieval.png', dpi=200)
plt.show()
print('Kaydedildi:', son / 'grafik-retrieval.png')

In [ ]:
# Soru tipine göre kırılım (RQ2): bicimbilimsel/paraphrase sorularda kayıp
ndcg = soru[soru['metrik'] == 'ndcg@10']
kirilim = ndcg.groupby(['sistem', 'tip'])['skor'].mean().unstack()
kirilim

In [ ]:
# Maliyet özeti (RQ4): gecikme ve indeks süresi
ozet[['sistem', 'indeks_suresi_sn', 'ort_gecikme_ms']]

## Raporlama kontrol listesi (İP-8)

- [ ] Her tabloda kullanılan konfigürasyon ve tohumlar belirtilmiş
- [ ] Sistem çiftleri için eşleştirilmiş bootstrap güven aralıkları eklenmiş
- [ ] Soru tipine göre kırılım (RQ2) ve maliyet analizi (RQ4) dahil
- [ ] Hata analizi: en kötü 10 soru elle incelenip sınıflandırılmış
- [ ] Sonuç klasörleri Drive'a yedeklenmiş ve repoya özet CSV'ler işlenmiş